# R10 · Does compact source-shape information help a full learned assessor?

This focused notebook keeps the original ARM answer **H** and compares the same full-v1 learner with three input conditions:

| Condition | Inputs | What changes |
|---|---|---|
| Full-v1 refit | All 28 existing team features, including SA inputs | Matched model-family control |
| Full-v1 + source shape | The same 28 + log reference count and proxy error from R10 | Additional child-distribution information |
| Full-v1 + matched fine | The same 28 + the same two reference features from an equal-sidecar-budget count grid | Whether a finer count reference explains the benefit |

All three use the existing log-error label floor 0.001, 63 leaves, 4,000-tree maximum and training-only early stopping. They are **new refits of the team model family on one fixed H and shared cell-group roles**, not the unavailable private Databricks model or the joint multi-merge-rate model from team notebook 12. Analytic R10, native R08, SA and the matched grid remain controls. The live Python adapter reconstructs histogram estimates; the original team v1 table instead receives some merge-shift inputs from Java-logged `est`/`srcSumEst`. This input contract can differ at floating-point precision and affect tree paths. The comparison does not establish that the original saved team model has the reported accuracy or deployment mismatch.

The study asks whether source-child moments contain useful incremental information. Hierarchical histograms, Beta distributions, copulas and ordinary source-TV allocation are established tools; this notebook does not establish world-first novelty or a universal 5% guarantee. Negative and positive controls must be reported together. Earlier research remains in Git history; running R01–R09 is unnecessary.


## Dependencies

Run this cell first. Missing packages are installed in the notebook environment; LightGBM is pinned to 4.6.0. After an installation restart, continue with the next cell. Restart Python after pulling changed implementation files so old imported modules are not reused.


In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys

missing = [name for name in ("numpy", "pandas", "scipy", "matplotlib")
           if importlib.util.find_spec(name) is None]
if importlib.util.find_spec("lightgbm") is None:
    missing.append("lightgbm==4.6.0")
elif importlib.metadata.version("lightgbm") != "4.6.0":
    raise RuntimeError("Use a notebook-scoped install of lightgbm==4.6.0, then restart Python and rerun this cell.")
if missing:
    if "dbutils" in globals():
        get_ipython().run_line_magic("pip", "install " + " ".join(missing))
        print("Installation finished. Restarting notebook Python; continue with the next cell.")
        dbutils.library.restartPython()
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
print("Required packages are available. LightGBM controls use version 4.6.0.")


## Paths and optional execution

`run_dir` is an actual histogram-export/result folder from the team pipeline, not the Git checkout. Alternatively set `histogram_dir`, `queries_file`, and `raw_data_dir` explicitly. `src_<id>.txt` and `arm_130.txt` must describe the same raw input. Change `base_resolution` if the chosen export uses another resolution, before creating a new experiment.

The grouped-cell split is inherited or generated by the audited recipe. Known AUTHORS inputs reproduce the recorded memberships byte-for-byte; another dataset creates a newly declared cohort, not the old measurements. Explicit ordinary query splits are not silently overwritten.

Outputs default outside Git. Blank resume fields create new unique directories; completed directories must stay at their original paths for hash-locked resume. `execution_backend=compiled` requires g++ and a compatible SciPy C API; select `indexed` explicitly otherwise. These Python-request timings are distinct from Java.

For the optional **shared Java engine**, supply `java_classpath` with a freshly built SynopsisLake jar/classes plus commons-math3, and `java_executable`. The new CLI supports `-shapeDir/-shapePacket` and explicit `-augmentShape true`. Java and Python feature calculations can produce different tree decisions, so Java uses its own original-calibration scores and freezes those gates **before** fresh labels. A blank classpath skips this optional profile. `run_java_timing` defaults to false because it loads and scans the actual original data in separate JVMs.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import importlib.metadata
import json
import os
import shutil
import sys


def detect_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    try:
        notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        here = Path("/Workspace") / notebook_path.lstrip("/")
        candidates += [here.parent, *here.parents]
    except Exception:
        pass
    for candidate in candidates:
        if (candidate / "ml" / "shape_inputs.py").is_file():
            return candidate.resolve()
    return Path.cwd().resolve()


def default_output_root(repo):
    try:
        user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
        return str(Path("/Workspace/Users") / user / "synopsislake-results" / "R10")
    except Exception:
        return str(repo.parent / "synopsislake-results" / "R10")


detected_repo = detect_repo_root()
defaults = {
    "repo_root": str(detected_repo), "run_dir": "", "histogram_dir": "",
    "raw_data_dir": str(detected_repo / "range_query"), "queries_file": "",
    "query_metadata_file": "", "split_file": "", "split_policy": "auto", "split_sha256": "", "base_resolution": "130",
    "output_root": default_output_root(detected_repo), "preparation_dir": "",
    "source_development_dir": "", "comparison_development_dir": "",
    "comparison_confirmation_dir": "", "java_profile_dir": "",
    "run_fresh_confirmation": "true", "run_execution": "true", "execution_backend": "compiled",
    "java_classpath": "", "java_executable": "java", "run_java_timing": "false",
}
try:
    for key, value in defaults.items():
        try:
            dbutils.widgets.get(key)
        except Exception:
            dbutils.widgets.text(key, value)
    settings = {key: dbutils.widgets.get(key).strip() for key in defaults}
except NameError:
    settings = dict(defaults)
repo_root = Path(settings["repo_root"]).expanduser().resolve()
if not (repo_root / "ml" / "shape_inputs.py").is_file():
    raise FileNotFoundError("Set repo_root to the checkout containing ml/shape_inputs.py.")
output_root = Path(settings["output_root"]).expanduser().resolve()
if output_root == repo_root or repo_root in output_root.parents:
    raise ValueError("output_root must be outside the Git checkout.")
output_root.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(output_root / "matplotlib-cache"))
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from ml import shape_inputs
from ml import local_shape_workflow as workflow
from ml import shape_execution
from ml import shape_team_comparison as comparison
from ml import sidecar_workflow as prior
import numpy as np
import pandas as pd


def boolean_setting(name):
    value = settings[name].lower()
    if value not in ("true", "false"):
        raise ValueError(name + " must be true or false")
    return value == "true"

backend = settings["execution_backend"]
if backend not in ("compiled", "indexed"):
    raise ValueError("execution_backend must be compiled or indexed")
if boolean_setting("run_execution") and backend == "compiled" and shutil.which("g++") is None:
    raise RuntimeError("Compiled execution requires g++. Install it in the runtime or explicitly select execution_backend=indexed.")
bundle_dir = output_root / ("R10-review-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid4().hex[:8])
bundle_dir.mkdir(exist_ok=False)
print("Repository:", repo_root)
print("Outputs:", output_root)
print("This review bundle:", bundle_dir)


## Prepare real inputs and grouped memberships

The input adapter verifies source cell masses against actual raw points. Stored offline truth is checked against fresh inclusive scans. No synthetic replacement dataset is used. The split recipe uses source-cell size and query midpoint; truth is used only for the declared offline eligibility cohort, never to choose groups or roles. Truth>=10 is not an online filter.


In [ ]:
input_config = {
    "repo_root": str(repo_root), "output_root": str(output_root),
    "run_dir": settings["run_dir"], "histogram_dir": settings["histogram_dir"],
    "raw_data_dir": settings["raw_data_dir"], "queries_file": settings["queries_file"],
    "query_metadata_file": settings["query_metadata_file"], "split_file": settings["split_file"],
    "split_kind": "cell", "base_resolution": int(settings["base_resolution"]),
    "preparation_dir": settings["preparation_dir"], "seeds": [0, 1, 2],
    "min_exact": 10, "chunk_size": 250000,
}
from ml import shape_split_inputs
input_config, split_handoff_manifest = shape_split_inputs.prepare_grouped_split(
    input_config, policy=settings["split_policy"], expected_sha256=settings["split_sha256"],
)
if split_handoff_manifest is not None:
    print("Grouped split handoff manifest:", split_handoff_manifest)
preparation_dir = shape_inputs.prepare_inputs(input_config)
preparation_dir, preparation_state = workflow._load_preparation(preparation_dir)
print("Audited preparation:", preparation_dir)
print("Real rows:", preparation_state["raw_rows"])
print("Native auxiliary bytes:", preparation_state["native_bytes"])
source_audit = pd.read_csv(preparation_dir / "source_audit.csv")
display(source_audit) if "display" in globals() else print(source_audit.to_string(index=False))

# The learned baseline also needs the original source text exports.
histogram_dir = str(input_config.get("histogram_dir") or "")
if not histogram_dir:
    inherited_config = dict(preparation_state["config"])
    inherited_config["repo_root"] = str(repo_root)
    inherited_config["run_dir"] = preparation_state.get("source_r09_run") or preparation_state.get("source_r08_run") or inherited_config.get("run_dir", "")
    histogram_dir = str(prior._resolve_inputs(prior._config(inherited_config))[0])
print("Original team feature source exports:", histogram_dir)


## Prepare the prespecified compact source representation

The 49,152-byte correction ceiling was selected in earlier TRAIN-only development and is fixed for this follow-up. Actual materialized bytes, native metadata and the matched count-grid bytes are reported. This source-only packet is not reallocated based on the new test. Only the new full-v1 learned comparison is trained here; old 8/12-feature tiny controls are unnecessary for this question. A hash-verified earlier source development may contain those extra controls; they are ignored by this common comparison. The same selected 49,152-byte source packet and source identities remain mandatory.


In [ ]:
if settings["source_development_dir"]:
    source_development = Path(settings["source_development_dir"]).expanduser().resolve()
    workflow.verify_protocol_lock(source_development)
    source_state = json.loads((source_development / "run.json").read_text())
    if (source_state.get("status") != "complete" or source_state["config"]["family"] != "hierarchical" or
        source_state["config"]["seeds"] != [0, 1, 2] or source_state["config"]["min_exact"] != 10 or
        source_state["config"]["epsilon"] != 0.25 or source_state["config"]["delta"] != 0.05):
        raise ValueError("Resume source differs from the declared family/seeds/error/cohort settings.")
    if Path(source_state["preparation_dir"]).resolve() != preparation_dir:
        raise ValueError("Source development must belong to this audited preparation.")
else:
    source_development = workflow.develop_experiment(dict(
        family="hierarchical", preparation_dir=str(preparation_dir), output_root=str(output_root),
        seeds=[0, 1, 2], train_queries=0, validation_queries=0,
        extra_budgets=[49152], selection_budgets=[49152], include_ml=False,
        evaluate_old_test=False, calibration_queries=512, test_queries=512,
        epsilon=0.25, delta=0.05, min_exact=10,
    ))
selection = json.loads((source_development / "selection.json").read_text())
if selection["selected_extra_budget"] != 49152:
    raise RuntimeError("The declared compact source representation did not survive TRAIN validation; do not force fresh confirmation.")
print("Frozen source development:", source_development)
print(json.dumps(selection, indent=2))


## Fit the common full-v1 comparison and freeze gates

Only inherited outer TRAIN rows fit the models. Every tenth sorted TRAIN qId is held out for stopping, exactly as the existing learner does. All three input conditions share the same outer rows, target, capacity and stopping policy. Original calibration selects the gates at epsilon=25%, delta=5%; test labels select no setting.

The primary count-grid reference matches the **analytic sidecar budget**. Learned weights and schemas have their own measured sizes and need not be identical across fitted models. Text source-histogram storage is also charged and is not claimed codec-optimal.


In [ ]:
comparison_config = dict(
    source_development_dir=str(source_development), histogram_dir=histogram_dir,
    preparation_dir=str(preparation_dir), output_root=str(output_root), seeds=[0, 1, 2],
    train_queries=0, calibration_queries=512, epsilon=0.25, delta=0.05, min_exact=10,
    fresh_query_count=2048, fresh_query_seed=2026111902,
    timing_queries=32 if boolean_setting("run_execution") else 0,
    timing_repeats=3, backend=backend, bootstrap_repeats=2000,
)
if settings["comparison_development_dir"]:
    comparison_development = Path(settings["comparison_development_dir"]).expanduser().resolve()
    saved_state, _ = comparison.verify_development(comparison_development)
    if Path(saved_state["preparation_dir"]).resolve() != preparation_dir:
        raise ValueError("Frozen comparison uses a different effective preparation.")
    for key, value in comparison_config.items():
        if key not in ("output_root", "preparation_dir") and saved_state["config"].get(key) != value:
            raise ValueError("Frozen comparison setting differs: " + key)
else:
    comparison_development = comparison.develop_experiment(comparison_config)
comparison.verify_development(comparison_development)
print("Frozen full-model development:", comparison_development)
print("Saved models:", comparison_development / "models")
print("Fixed gates:", comparison_development / "frozen_gate_cutoffs.csv")


## Optional Java profile: freeze before generating fresh truth

This calls the existing `ApproxQueryExp` engine for SA, full-v1, R10 and full-v1+R10. It first calibrates using the same original calibration IDs, then persists new geometry and every Java score without labels. It does not replace the Python gates or claim numerical equivalence. Full-v1+fine remains a Python scientific control until a fine-reference Java adapter is supplied.

Transfer checks of this reconstructed-input refit found that very small merge-shift rounding differences can change learned tree paths. The original team training table uses Java-origin estimates, so this is not an established defect in its original pipeline. Java's calibration is therefore a separately reported backend condition, not evidence that R10 beats a broken or identical Python model. Exact/fallback Java timing is optional after independent truth is available.


In [ ]:
java_profile = None
if settings["java_classpath"] or settings["java_profile_dir"]:
    from ml import shape_java_comparison as java_comparison
    if settings["java_profile_dir"]:
        java_profile = Path(settings["java_profile_dir"]).expanduser().resolve()
        java_state, _ = java_comparison.verify_profile(java_profile)
        if Path(java_state["config"]["development_dir"]).resolve() != comparison_development:
            raise ValueError("Java profile belongs to another comparison development.")
    else:
        if settings["comparison_confirmation_dir"]:
            raise ValueError("A new Java profile must precede fresh labels. Resume the original profile instead.")
        java_profile = java_comparison.prepare_profile(dict(
            development_dir=str(comparison_development), output_root=str(output_root),
            repo_root=str(repo_root), histogram_dir=histogram_dir,
            java=settings["java_executable"], java_classpath=settings["java_classpath"], heap="1g",
        ))
    print("Java scores frozen without labels:", java_profile)
else:
    print("Java profile skipped: no explicit classpath. The Python scientific comparison remains enabled.")


## Fresh confirmation and optional live timing

The cohort contains 2,048 rectangles from seed 2026111902. Width/height donors come only from original seed-0 TRAIN geometry. The same rectangles serve every method and seed; these are not 6,144 independent queries. Repeating the seed reproduces this cohort.

Every model/packet/gate freezes before geometry, and every prediction precedes exact labels. Three independent original-row scans audit indexed labels. If enabled, Python live timing selects 32 IDs per seed before labels, with three repetitions and resident/reload payload modes. Every exact request or rejected answer scans all verified resident raw points; saved truth is never the timed answer source.


In [ ]:
comparison_confirmation = None
if boolean_setting("run_fresh_confirmation"):
    if settings["comparison_confirmation_dir"]:
        comparison_confirmation = Path(settings["comparison_confirmation_dir"]).expanduser().resolve()
        saved_state, _ = comparison.verify_confirmation(comparison_confirmation)
        if (saved_state.get("status") != "complete" or
            Path(saved_state["development_dir"]).resolve() != comparison_development):
            raise ValueError("Resume requires a completed matching confirmation.")
    else:
        comparison_confirmation = comparison.confirm_experiment(dict(
            comparison_config, phase="confirm", development_dir=str(comparison_development),
        ))
    print("Completed fresh comparison:", comparison_confirmation)
    if java_profile is not None:
        java_comparison.finalize_profile(dict(profile_dir=str(java_profile),
            confirmation_dir=str(comparison_confirmation)))
        if boolean_setting("run_java_timing"):
            java_comparison.run_timing(dict(profile_dir=str(java_profile)))
        print("Separate Java quality/profile:", java_profile)
else:
    print("Fresh confirmation disabled; development evidence is not a test result.")


## Read quality, storage and time together

MAE is the difference between predicted error and the true error of unchanged H, in percentage points. It is not answer accuracy. SA is a heuristic score, so its numeric error-prediction MAE is undefined. AUC and fixed-coverage bad fractions describe ranking. Zero accepted answers have undefined accepted risk.

The truth>=10 numerical cohort avoids infinite relative errors at zero truth, but all/small/zero counts and risks remain present. The displayed macro seed means reuse the same test geometry; they are not independent datasets. Paired intervals first average seed appearances per query and use fixed spatial blocks; they are conditional descriptive intervals.

Latency pools actual observations and uses exact-mean/method-mean speedup. P95 comes from pooled observations, not averages of per-seed quantiles. Read timing-subset quality separately from the full fresh cohort. Initialization and compilation are separate; no claim of free setup, cloud performance or universal risk follows.


In [ ]:
def show_table(frame):
    display(frame) if "display" in globals() else print(frame.to_string(index=False))

show_table(pd.read_csv(comparison_development / "payload_costs.csv"))
if comparison_confirmation is not None:
    metrics = pd.read_csv(comparison_confirmation / "metrics.csv")
    show_table(metrics)
    show_table(pd.read_csv(comparison_confirmation / "paired_mae_intervals.csv"))
    summary = metrics[metrics.cohort == "truth_ge_10"].groupby("method", sort=False).agg(
        mae_pp=("error_prediction_mae_pp", "mean"), auc=("bad_answer_auc", "mean"),
        accepted=("accepted_count", "mean"), accepted_bad=("accepted_bad_count", "mean"),
        acceptance=("acceptance", "mean"), accepted_violation=("accepted_violation", "mean"),
        configurations_over_delta=("accepted_violation", lambda x: int((x > 0.05).sum())))
    show_table(summary.reset_index())
    summary.to_csv(bundle_dir / "fresh_primary_macro.csv")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
    summary.mae_pp.dropna().plot.barh(ax=ax)
    ax.set_xlabel("Error-prediction MAE, percentage points; truth >=10")
    fig.savefig(bundle_dir / "prediction_mae.png", dpi=160)
    plt.show()
    plt.close(fig)
    counts = metrics[metrics.cohort == "all"].groupby("method", sort=False)[
        ["accepted_good_count", "accepted_bad_count"]].mean()
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
    counts.plot.barh(stacked=True, ax=ax, color=["#328b74", "#cf6750"])
    ax.set_xlabel("Mean accepted query count across seeds; shared 2,048-query cohort")
    fig.savefig(bundle_dir / "accepted_good_and_bad.png", dpi=160)
    plt.show()
    plt.close(fig)
    curves = pd.read_csv(comparison_confirmation / "fixed_coverage.csv")
    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    for method in ("ML_v1", "ML_v1_shape", "ML_v1_fine", "R10", "SA"):
        selected = curves[(curves.cohort == "all") & (curves.method == method)]
        line = selected.groupby("nominal_coverage").bad_fraction.mean()
        ax.plot(line.index*100, line.values*100, "o-", label=method)
    ax.set(xlabel="Nominal fraction accepted (%)", ylabel="Bad fraction among accepted (%)")
    ax.legend()
    fig.savefig(bundle_dir / "equal_coverage_bad_fraction.png", dpi=160)
    plt.show()
    plt.close(fig)
    timing = comparison_confirmation / "e2e_pooled.csv"
    if timing.is_file():
        show_table(pd.read_csv(timing))
if java_profile is not None:
    for filename in ("java_metrics.csv", "fresh_backend_parity.csv", "java_timing_summary.csv"):
        file = java_profile / filename
        if file.is_file():
            print("Separate Java backend:", filename)
            show_table(pd.read_csv(file))


## Export review evidence, not raw data or a hidden model lookup

The small review ZIP includes per-query scores, labels, gates, counts, metrics, protocols, costs and source snapshots. Model weights remain in `comparison_development/models/seed_<s>/...`; their schemas and hashes are included but large model text files are excluded from this upload packet. The models are still charged in storage tables. Original points, caches, nested archives and compiled binaries are excluded. Existing runs are never overwritten.

This notebook is an incremental-information and integration experiment. An outcome only better than full-v1, but not better than full-v1+fine, does not demonstrate a special benefit from child moments. An outcome changing across datasets or budgets is a limitation to report. Source grids still grow with source count; arbitrary bounded merging is unsolved. Do not select a rule from these test outcomes and claim improvement on the same cohort.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

phases = {"source_development": source_development,
          "common_development": comparison_development}
if comparison_confirmation is not None:
    phases["common_confirmation"] = comparison_confirmation
if java_profile is not None:
    phases["java_profile"] = java_profile
manifest = dict(paths={name: str(path) for name, path in phases.items()},
    versions={name: importlib.metadata.version(name) for name in
              ("numpy", "pandas", "scipy", "matplotlib", "lightgbm", "scikit-learn")},
    same_dataset_fresh_seed=2026111902, planned_unique_queries=2048,
    actual_unique_queries=(int(pd.read_csv(comparison_confirmation / "queries_geometry_only.csv").qId.nunique())
                           if comparison_confirmation is not None else 0),
    independent_dataset=False, repeated_seeds_are_independent=False,
    source_information_comparison="full28 versus full28+R10 and full28+matched fine",
    global_novelty_established=False, hard_five_percent_guarantee=False)
prior._json(bundle_dir / "manifest.json", manifest)
archive_path = bundle_dir / "R10_comparison_results.zip"
with ZipFile(archive_path, "w", ZIP_DEFLATED) as archive:
    for name, root in phases.items():
        for file in sorted(root.rglob("*")):
            relative = file.relative_to(root)
            if (not file.is_file() or file.suffix in (".zip", ".so", ".pyc") or
                set(relative.parts) & {"raw_cache", "raw_timing", "feature_cache", "__pycache__", "confirmation", "development"} or
                (file.name == "model.txt" and "models" in relative.parts)):
                continue
            archive.write(file, str(Path(name) / relative))
    if split_handoff_manifest is not None:
        archive.write(split_handoff_manifest, "split_handoff_manifest.json")
    for file in sorted((repo_root / "ml").glob("*")):
        if file.is_file() and file.suffix in (".py", ".cc", ".json"):
            archive.write(file, str(Path("implementation/ml") / file.name))
    for relative in (
        "predictor/SourceShapePredictor.java", "predictor/SourceAugmentedLearnedPredictor.java",
        "predictor/PredictorFactory.java", "sigspatial25/ApproxQueryExp.java",
        "sigspatial25/SourceShapeReferenceExp.java",
    ):
        file = repo_root / "SynopsisLake/src/main/java" / relative
        archive.write(file, str(Path("implementation/java") / relative))
    for file in sorted((repo_root / "tests/fixtures").glob("*")):
        if file.is_file() and file.suffix in (".py", ".json"):
            archive.write(file, str(Path("implementation/tests/fixtures") / file.name))
    archive.write(repo_root / "ml/notebooks/R10_source_shape_assessment.ipynb",
                  "implementation/ml/notebooks/R10_source_shape_assessment.ipynb")
    for file in sorted(bundle_dir.iterdir()):
        if file.is_file() and file.suffix != ".zip":
            archive.write(file, file.name)
print("Share this review packet:", archive_path)
print("Models remain here:", comparison_development / "models")
print("Resume paths and versions:", bundle_dir / "manifest.json")
